# SpendDNA — Your Wallet’s Six-Month Story


**Student:** Arathi A  
**Batch:** SEPTEMBER BATCH 2026
**Prepared:**28 september 2026  
**Dataset:** `rahul_transactions.csv`  
**Stack:** Python, pandas, NumPy, datetime — no regex, no ML, no plotting libraries.

**Problem:** Clean mixed-format transaction data, normalize merchants, classify purchases, analyze timing and monthly trends, identify large within-category outliers, and match financial archetypes. The final text-only report is built from computed numbers, not hard-coded example values.

> **Run:** Upload the notebook and the accompanying `rahul_transactions.csv` to the same Google Colab session; choose **Runtime → Run all**. If the CSV cannot be found, the first code cell prompts you to upload it. This dataset is synthetic; never publish real bank statements.

## Feature 1 — Transaction parser and data-quality check
Handles four explicit Indian/ISO date formats, multiple currency notations, debit/credit aliases, duplicate transactions and missing modes. Explicit formats avoid the ambiguous behavior of newer pandas when parsing a column containing mixed date formats.

In [6]:
# Reviewed and adapted Python implementation; verify results and explain each calculation.
import numpy as np
import pandas as pd
from datetime import datetime
from pathlib import Path

DATA_PATH = Path('/content/rahul_transactions.csv')
if not DATA_PATH.exists():
    try:
        from google.colab import files
        print('Upload rahul_transactions.csv to continue.')
        uploaded = files.upload()
        if 'rahul_transactions.csv' not in uploaded:
            raise FileNotFoundError('Select the exact file rahul_transactions.csv')
    except ImportError:
        raise FileNotFoundError('Place rahul_transactions.csv in the notebook working directory.')

raw = pd.read_csv(DATA_PATH, dtype=str, keep_default_na=False)
required_columns = ['Date', 'Time', 'Description', 'Type', 'Amount', 'Balance', 'Mode', 'Ref']
assert all(name in raw.columns for name in required_columns), 'CSV is missing one or more required columns.'
original_rows = len(raw)
df_clean = raw.drop_duplicates().copy()
duplicates_removed = original_rows - len(df_clean)

DATE_FORMATS = ('%d/%m/%y', '%Y-%m-%d', '%d-%b-%y', '%d %b %Y')
def parse_bank_date(text):
    """Parse an individual string with the four allowed formats, without guessing DD/MM."""
    for date_format in DATE_FORMATS:
        parsed = pd.to_datetime(str(text).strip(), format=date_format, errors='coerce', dayfirst=True)
        if not pd.isna(parsed):
            return parsed
    return pd.NaT

amount_text = (df_clean['Amount'].astype(str).str.strip()
    .str.replace('₹', '', regex=False)
    .str.replace('Rs.', '', regex=False)
    .str.replace('RS.', '', regex=False)
    .str.replace(',', '', regex=False).str.strip())
df_clean['amount'] = pd.to_numeric(amount_text, errors='coerce')
df_clean['date'] = df_clean['Date'].apply(parse_bank_date)
df_clean['type'] = (df_clean['Type'].str.strip().str.lower()
    .replace({'dr': 'debit', 'cr': 'credit'}))
df_clean['Mode'] = df_clean['Mode'].replace('', np.nan)

invalid_dates = int(df_clean['date'].isna().sum())
invalid_amounts = int(df_clean['amount'].isna().sum())
invalid_types = int((~df_clean['type'].isin(['debit', 'credit'])).sum())
df_clean = df_clean.dropna(subset=['date', 'amount']).copy()
df_clean = df_clean[df_clean['type'].isin(['debit', 'credit'])].copy()
df_clean['hour'] = pd.to_numeric(df_clean['Time'].str[:2], errors='coerce')
df_clean = df_clean[df_clean['hour'].between(0, 23)].copy()
df_clean['hour'] = df_clean['hour'].astype(int)
df_clean['month'] = df_clean['date'].dt.month
df_clean['month_name'] = df_clean['date'].dt.strftime('%b')
df_clean['day_of_week'] = df_clean['date'].dt.day_name()

print(f'Loaded {original_rows:,} rows; removed {duplicates_removed} exact duplicates.')
print(f'Valid transactions: {len(df_clean):,}; date failures: {invalid_dates}; amount failures: {invalid_amounts}; type failures: {invalid_types}.')
print(f'Coverage: {df_clean.date.min():%d %b %Y} through {df_clean.date.max():%d %b %Y}; {df_clean.month.nunique()} months.')
print('Normalized transaction types:', df_clean['type'].value_counts().to_dict())
assert duplicates_removed == 18, 'Expected 18 duplicate rows in supplied dataset.'
assert len(df_clean) == 1310, 'Expected 1310 clean rows with the supplied synthetic dataset.'

Loaded 1,328 rows; removed 18 exact duplicates.
Valid transactions: 1,310; date failures: 0; amount failures: 0; type failures: 0.
Coverage: 01 Jan 2024 through 30 Jun 2024; 6 months.
Normalized transaction types: {'debit': 1304, 'credit': 6}


## Feature 2 — Canonical vendor extractor
Keyword lists are ordered so specific patterns (e.g., **Instamart**, **Amazon Prime**, **Zomato Dining**) are checked **before** their broader parent brands. Transfers and ATM transactions receive their own canonical labels. Inspect the audit output to ensure unknown descriptions are not quietly assigned to arbitrary categories.

In [7]:
# Reviewed and adapted Python implementation; verify results and explain each calculation.
# These merchant aliases were checked against the description values in the supplied CSV.
# Specific aliases must come before generic aliases. All matching uses Python string methods, no regex.
MERCHANT_ALIASES = {
    'Salary': ['NEFT-TECHCRUSH', 'SALARY MAY24'],
    'Rent': ['IMPS-RENT-LANDLORD'],
    'Cash Withdrawal': ['ATM-WDL-'],
    'Personal Transfer': ['UPI-ANKIT-', 'UPI-PRIYA-', 'UPI-AMAN-', 'UPI-NEHA-', 'UPI-SNEHA-', 'UPI-KARAN-', 'UPI-VIKAS-'],
    'Instamart': ['INSTAMART'],
    'Amazon Prime': ['AMAZON PRIME', 'AMZN PRIME', 'UPI-AMAZON-PRIME'],
    'Zomato Dining': ['ZOMATO-DINING'],
    'BookMyShow': ['BOOKMYSHOW', 'BIGTREE ENTERTAINMENT', 'BMS MOVIE TICKETS'],
    'Disney+ Hotstar': ['HOTSTAR', 'STAR INDIA PVT'],
    'Netflix': ['NETFLIX'],
    'Spotify': ['SPOTIFY'],
    'Swiggy': ['SWIGGY', 'BUNDL'],
    'Zomato': ['ZOMATO'],
    'Zepto': ['ZEPTO', 'KIRANAKART'],
    'Blinkit': ['BLINKIT', 'GROFERS'],
    'BigBasket': ['BIGBASKET', 'INNOVATIVE RETAIL'],
    'D-Mart': ['DMART', 'AVENUE SUPERMARTS'],
    'Amazon': ['AMAZON', 'AMZN'],
    'Flipkart': ['FLIPKART', 'FKART'],
    'Myntra': ['MYNTRA'],
    'Nykaa': ['NYKAA', 'FSN E-COMMERCE'],
    'Uber': ['UBER'],
    'Ola': ['OLA CABS', 'OLA-PRIME', 'OLA ELECTRIC', 'OLACABS', 'ANI TECHNOLOGIES'],
    'Rapido': ['RAPIDO', 'ROPPEN TRANSPORTATION'],
    'BMTC': ['BMTC', 'TUMMOC'],
    'Starbucks': ['STARBUCKS'],
    'Third Wave Coffee': ['THIRD WAVE', 'THIRDWAVE', 'TWC INDIA'],
    'Cafe Coffee Day': ['COFFEE DAY', 'UPI-CCD'],
    'Meghana Foods': ['MEGHANA FOODS'],
    'Empire Restaurant': ['EMPIRE RESTAURANT'],
    'Truffles': ['TRUFFLES'],
    'Dineout': ['DINEOUT'],
    'Bangalore Restaurant': ['BANGALORE RESTAURANT'],
    'Other Restaurant': ['RESTAURANT'],
    'BESCOM': ['BESCOM', 'BANGALORE ELEC SUPPLY'],
    'BWSSB': ['BWSSB'],
    'Jio': ['JIO'],
    'Airtel': ['AIRTEL', 'BHARTI AIRTEL'],
    'Vi': ['VI POSTPAID', 'VODAFONE IDEA', 'UPI-VI-'],
    'Zerodha': ['ZERODHA'],
    'Groww': ['GROWW', 'NEXTBILLION'],
    'Indian Oil': ['INDIAN OIL', 'UPI-IOC'],
    'HP Petrol': ['HP PETROL'],
    'BPCL': ['BPCL'],
}

def extract_vendor(description):
    upper = str(description).upper().strip()
    for canonical_vendor, keywords in MERCHANT_ALIASES.items():
        if any(keyword in upper for keyword in keywords):
            return canonical_vendor
    return 'Uncategorised'

df_clean['vendor_clean'] = df_clean['Description'].map(extract_vendor)
unknown_descriptions = (df_clean.loc[df_clean.vendor_clean.eq('Uncategorised'), 'Description']
    .value_counts())
print(f'Canonical vendor labels: {df_clean.vendor_clean.nunique()}')
print('Top 10 vendors by frequency:')
print(df_clean.vendor_clean.value_counts().head(10).to_string())
print(f'Unmapped transactions: {len(df_clean[df_clean.vendor_clean.eq("Uncategorised")])}; unique unmapped descriptions: {len(unknown_descriptions)}')
if len(unknown_descriptions):
    print(unknown_descriptions.to_string())

Canonical vendor labels: 44
Top 10 vendors by frequency:
vendor_clean
Swiggy       176
Zomato       101
Ola           87
Amazon        76
Zepto         71
Uber          71
Instamart     67
Blinkit       55
Rapido        55
Flipkart      47
Unmapped transactions: 0; unique unmapped descriptions: 0


## Feature 3 — Category tagger
The brief lists 12 core spending categories, but the file also contains salary, rent, person-to-person transfers and cash withdrawals. These are kept in separate, explicitly named buckets, instead of mislabeling rent as shopping or transfers as purchases. The share-of-debits calculations make their denominator explicit.

In [8]:
# Reviewed and adapted Python implementation; verify results and explain each calculation.
CATEGORIES = {
    'Food Delivery': ['Swiggy', 'Zomato'],
    'Quick Commerce': ['Instamart', 'Zepto', 'Blinkit'],
    'E-commerce': ['Amazon', 'Flipkart', 'Myntra', 'Nykaa'],
    'Transport': ['Uber', 'Ola', 'Rapido', 'BMTC'],
    'Cafe': ['Starbucks', 'Third Wave Coffee', 'Cafe Coffee Day'],
    'Restaurants': ['Zomato Dining', 'Meghana Foods', 'Empire Restaurant', 'Truffles', 'Dineout', 'Bangalore Restaurant', 'Other Restaurant'],
    'Subscriptions': ['Netflix', 'Amazon Prime', 'Disney+ Hotstar', 'Spotify'],
    'Utilities': ['BESCOM', 'BWSSB', 'Jio', 'Airtel', 'Vi'],
    'Groceries': ['BigBasket', 'D-Mart'],
    'Investments': ['Zerodha', 'Groww'],
    'Fuel': ['Indian Oil', 'HP Petrol', 'BPCL'],
    'Entertainment': ['BookMyShow'],
    'Housing': ['Rent'],                  # Necessary additional class: monthly rent is not a utility.
    'Personal Transfer': ['Personal Transfer'],
    'Cash Withdrawal': ['Cash Withdrawal'],
    'Income': ['Salary'],
}
VENDOR_TO_CATEGORY = {
    merchant: category
    for category, merchants in CATEGORIES.items()
    for merchant in merchants
}
df_clean['category'] = df_clean['vendor_clean'].map(VENDOR_TO_CATEGORY).fillna('Uncategorised')
print('All category frequencies:')
print(df_clean.category.value_counts().to_string())
print('Uncategorised rows:', int(df_clean.category.eq('Uncategorised').sum()))
assert df_clean['category'].eq('Uncategorised').sum() < 5, 'Too many unclassified transactions; extend merchant aliases.'

All category frequencies:
category
Food Delivery        277
Transport            250
Quick Commerce       193
E-commerce           162
Cafe                  99
Restaurants           93
Utilities             43
Groceries             41
Subscriptions         41
Fuel                  28
Investments           23
Personal Transfer     18
Cash Withdrawal       17
Entertainment         13
Income                 6
Housing                6
Uncategorised rows: 0


## Feature 4 — Spending overview
Credits are salary inflows; debits are money leaving the bank account. Net change is **credits minus all debits**, which is a cash-flow measure, not a formal profit-and-loss measure. P2P transfers and ATM withdrawals remain in total cash outflows, but are left out of consumer merchant rankings.

In [17]:
# Reviewed and adapted Python implementation; verify results and explain each calculation.
debits = df_clean[df_clean['type'].eq('debit')].copy()
credits = df_clean[df_clean['type'].eq('credit')].copy()
amount_in = float(credits['amount'].sum())
amount_out = float(debits['amount'].sum())
net_cash_flow = amount_in - amount_out
savings_rate = (100 * net_cash_flow / amount_in) if amount_in else np.nan
NON_CONSUMPTION = {'Personal Transfer', 'Cash Withdrawal'}
merchant_spend = debits[~debits['category'].isin(NON_CONSUMPTION)].copy()
by_category = merchant_spend.groupby('category')['amount'].sum().sort_values(ascending=False)
by_vendor = merchant_spend.groupby('vendor_clean')['amount'].agg(['sum', 'count']).sort_values('sum', ascending=False)
share_of_debits = (by_category / amount_out * 100) if amount_out else by_category.copy()

print(f'Total credits : Rs. {amount_in:,.2f}')
print(f'Total debits  : Rs. {amount_out:,.2f}')
print(f'Net change    : Rs. {net_cash_flow:,.2f}')
print(f'Net cash-flow ratio : {savings_rate:,.1f}%')
print('Top 5 categories by cash outflow:')
print(by_category.head(5).round(2).to_string())
print('Top 5 vendors by cash outflow:')
print(by_vendor.head(5).round(2).to_string())

Total credits : Rs. 509,774.00
Total debits  : Rs. 1,678,901.00
Net change    : Rs. -1,169,127.00
Net cash-flow ratio : -229.3%
Top 5 categories by cash outflow:
category
E-commerce       593769.0
Investments      248160.0
Restaurants      127290.0
Food Delivery    119501.0
Housing          108000.0
Top 5 vendors by cash outflow:
                   sum  count
vendor_clean                 
Amazon        318422.0     76
Zerodha       210000.0     14
Flipkart      177510.0     47
Rent          108000.0      6
Swiggy         73738.0    176


## Feature 5 — Monthly trend matrix
A category × month pivot shows debits from January to June. Growth calculations compare June with January and skip categories having a zero January base, so percentages are not artificially infinite.

In [18]:
# Reviewed and adapted Python implementation; verify results and explain each calculation.
month_index = [1, 2, 3, 4, 5, 6]
month_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun']
monthly_spend = (debits.pivot_table(index='category', columns='month', values='amount',
                                   aggfunc='sum', fill_value=0)
    .reindex(columns=month_index, fill_value=0))
monthly_spend.columns = month_labels
change_amount = monthly_spend['Jun'] - monthly_spend['Jan']
valid_start = monthly_spend['Jan'] > 0
change_percent = (change_amount[valid_start] / monthly_spend.loc[valid_start, 'Jan'] * 100).sort_values()
monthly_debit_total = debits.groupby('month')['amount'].sum().reindex(month_index, fill_value=0)
print('Category-by-month spend (Rs.):')
print(monthly_spend.round(0).to_string())
print('\nMonthly debit totals:')
print(pd.Series(monthly_debit_total.to_numpy(), index=month_labels).round(0).to_string())
if len(change_percent):
    print(f'Biggest Jan-to-Jun % increase : {change_percent.idxmax()} ({change_percent.max():+.1f}%)')
    print(f'Biggest Jan-to-Jun % decrease : {change_percent.idxmin()} ({change_percent.min():+.1f}%)')

Category-by-month spend (Rs.):
                       Jan      Feb       Mar      Apr      May       Jun
category                                                                 
Cafe                3690.0   4273.0    5448.0   6564.0   5668.0    5802.0
Cash Withdrawal     2000.0   5000.0    8000.0   5500.0   8000.0   17000.0
E-commerce         97134.0  92773.0  103772.0  68098.0  95001.0  136991.0
Entertainment       1263.0    474.0    2418.0   2224.0      0.0    1914.0
Food Delivery      20206.0  18714.0   19166.0  20726.0  21193.0   19496.0
Fuel               30322.0   2079.0   26164.0  18718.0   9138.0    2882.0
Groceries          17649.0   8571.0    6289.0  11748.0   9718.0    5432.0
Housing            18000.0  18000.0   18000.0  18000.0  18000.0   18000.0
Investments        38432.0  15000.0   68644.0  54126.0  48628.0   23330.0
Personal Transfer   7852.0   4285.0    4275.0    663.0   3412.0    4112.0
Quick Commerce     12797.0  17465.0   17979.0  16572.0  15188.0   15666.0
Restaur

## Feature 6 — Time-of-day heatmap (text only)
An hourly frequency matrix is built from debits. **Late-night** means times from 21:00 inclusive to 02:00 exclusive (hours 21, 22, 23, 00 and 01). This definition is used consistently in both the report and archetype rules.

In [19]:
# Reviewed and adapted Python implementation; verify results and explain each calculation.
hour_count = (debits.pivot_table(index='category', columns='hour', values='amount',
                                 aggfunc='size', fill_value=0)
    .reindex(columns=list(range(24)), fill_value=0))
food_orders = debits[debits.category.eq('Food Delivery')]
late_orders = food_orders[(food_orders.hour >= 21) | (food_orders.hour < 2)]
late_order_pct = (100 * len(late_orders) / len(food_orders)) if len(food_orders) else 0.0
late_food_spend_pct = (100 * late_orders.amount.sum() / food_orders.amount.sum()) if food_orders.amount.sum() else 0.0
cafe_orders = debits[debits.category.eq('Cafe')]
morning_cafe_orders = cafe_orders[cafe_orders.hour.between(8, 11)]
morning_cafe_pct = 100 * len(morning_cafe_orders) / len(cafe_orders) if len(cafe_orders) else 0
print('Category x hour (number of debit transactions):')
print(hour_count.to_string())
print(f'\nFood delivery 21:00-01:59: {len(late_orders)}/{len(food_orders)} orders = {late_order_pct:.1f}% of orders; {late_food_spend_pct:.1f}% of food-delivery rupees.')
print(f'Cafe 08:00-11:59: {len(morning_cafe_orders)}/{len(cafe_orders)} orders = {morning_cafe_pct:.1f}% of orders.')

Category x hour (number of debit transactions):
hour               0   1   2   3   4   5   6   7   8   9   10  11  12  13  14  15  16  17  18  19  20  21  22  23
category                                                                                                         
Cafe                3   3   0   1   0   0   1   2   9   7  13   6   4   4   5   7   8  10   7   2   5   0   0   2
Cash Withdrawal     0   0   0   0   0   0   0   0   2   3   2   2   0   0   2   0   1   0   1   0   1   1   2   0
E-commerce          9   5   4   7   7   7   6   4   4   9   6  11   7   5   9   5   8   7   9   4   7   7   6   9
Entertainment       1   1   0   0   0   0   0   1   2   0   0   0   0   0   0   0   2   0   2   1   2   0   1   0
Food Delivery       1   7   2   4   6   7   1   1   7  10   5  18  19  11   9  13   9  10  27  29  34  19  19   9
Fuel                2   1   0   1   1   1   0   0   1   2   0   2   3   1   1   2   3   0   1   2   1   2   0   1
Groceries           2   1   2   6   1   

## Feature 7 — Within-category z-score anomaly detection
For each **debit category**, compute (amount − category mean) / sample standard deviation. Flag **z > 2**. This is an outlier indicator, not evidence of fraud. Categories with zero or undefined standard deviation cannot generate anomalies.

In [20]:
# Reviewed and adapted Python implementation; verify results and explain each calculation.
category_mean = debits.groupby('category')['amount'].transform('mean')
category_std = debits.groupby('category')['amount'].transform('std').replace(0, np.nan)
debits['z_score'] = ((debits['amount'] - category_mean) / category_std).fillna(0.0)
anomalies = debits[debits.z_score > 2].sort_values('z_score', ascending=False).copy()
print(f'Anomalies with within-category z > 2: {len(anomalies)}')
print(anomalies[['date', 'vendor_clean', 'category', 'amount', 'z_score']].head(10).to_string(index=False, formatters={'date': lambda x: x.strftime('%d %b %Y'), 'amount': lambda x: f'{x:,.0f}', 'z_score': lambda x: f'{x:.2f}'}))

Anomalies with within-category z > 2: 24
       date         vendor_clean    category amount z_score
26 Feb 2024 Bangalore Restaurant Restaurants  8,383    4.35
22 Jun 2024              Dineout Restaurants  7,935    4.07
31 Mar 2024        Meghana Foods Restaurants  7,931    4.07
26 Jun 2024               Amazon  E-commerce 22,008    3.97
07 Feb 2024               Amazon  E-commerce 21,986    3.97
04 Mar 2024             Truffles Restaurants  7,441    3.76
07 Jan 2024 Bangalore Restaurant Restaurants  7,314    3.69
05 Mar 2024               Amazon  E-commerce 19,917    3.52
02 Mar 2024               Amazon  E-commerce 18,273    3.17
27 May 2024             Flipkart  E-commerce 17,831    3.07


## Feature 8 — Spending archetype classification
Every rule is evaluated independently and the user may match multiple archetypes. The brief describes "eight" archetypes but its table includes **nine named rules**, so all nine are implemented. The subscriptions rule strictly requires **at least five distinct vendors**; it is not relaxed just to force an expected label.

In [21]:
# Reviewed and adapted Python implementation; verify results and explain each calculation.
def category_pct(category_names):
    subset = debits[debits.category.isin(category_names)]
    return float(subset.amount.sum() / amount_out * 100) if amount_out else 0.0

def rule_foodie():
    percent = category_pct(['Food Delivery', 'Restaurants', 'Cafe'])
    return percent > 25, f'{percent:.1f}% of debits (threshold >25%)'

def rule_quick_commerce():
    percent = category_pct(['Quick Commerce'])
    return percent > 15, f'{percent:.1f}% of debits (threshold >15%)'

def rule_shopaholic():
    percent = category_pct(['E-commerce'])
    return percent > 15, f'{percent:.1f}% of debits (threshold >15%)'

def rule_investor():
    percent = category_pct(['Investments'])
    return percent > 15, f'{percent:.1f}% of debits (threshold >15%)'

def rule_late_night_snacker():
    return late_order_pct > 50, f'{late_order_pct:.1f}% of food orders between 21:00 and 01:59'

def rule_cab_commuter():
    percent = category_pct(['Transport'])
    return percent > 10, f'{percent:.1f}% of debits (threshold >10%)'

def rule_subscription_lover():
    count = int(debits.loc[debits.category.eq('Subscriptions'), 'vendor_clean'].nunique())
    return count >= 5, f'{count} distinct subscription vendors (threshold >=5)'

def rule_yolo_spender():
    return bool(amount_in and savings_rate < 10), f'{savings_rate:.1f}% savings rate (threshold <10%)'

def rule_disciplined_saver():
    return bool(amount_in and savings_rate > 40), f'{savings_rate:.1f}% savings rate (threshold >40%)'

ARCHETYPE_RULES = {
    'THE FOODIE': rule_foodie,
    'THE QUICK COMMERCE JUNKIE': rule_quick_commerce,
    'THE SHOPAHOLIC': rule_shopaholic,
    'THE INVESTOR': rule_investor,
    'THE LATE-NIGHT SNACKER': rule_late_night_snacker,
    'THE CAB COMMUTER': rule_cab_commuter,
    'THE SUBSCRIPTION LOVER': rule_subscription_lover,
    'THE YOLO SPENDER': rule_yolo_spender,
    'THE DISCIPLINED SAVER': rule_disciplined_saver,
}
assessments = {name: check() for name, check in ARCHETYPE_RULES.items()}
active_archetypes = {name: support for name, (matched, support) in assessments.items() if matched}
print('Matched archetypes:')
for name, metric in active_archetypes.items():
    print(f'  {name:<29} {metric}')
print('\nRules not matched:')
for name, (matched, metric) in assessments.items():
    if not matched:
        print(f'  {name:<29} {metric}')

Matched archetypes:
  THE SHOPAHOLIC                35.4% of debits (threshold >15%)
  THE YOLO SPENDER              -229.3% savings rate (threshold <10%)

Rules not matched:
  THE FOODIE                    16.6% of debits (threshold >25%)
  THE QUICK COMMERCE JUNKIE     5.7% of debits (threshold >15%)
  THE INVESTOR                  14.8% of debits (threshold >15%)
  THE LATE-NIGHT SNACKER        19.9% of food orders between 21:00 and 01:59
  THE CAB COMMUTER              3.4% of debits (threshold >10%)
  THE SUBSCRIPTION LOVER        4 distinct subscription vendors (threshold >=5)
  THE DISCIPLINED SAVER         -229.3% savings rate (threshold >40%)


## Bonus — Data audit, day-of-week behavior, original archetype, next-month forecast
Bonus tasks use only pandas/NumPy and derive their values directly from the CSV. The invented archetype is **THE MARKETPLACE POWER USER**, triggered when Amazon and Flipkart together have at least 100 debit transactions **and** represent more than 25% of all debits. It captures repeated Indian marketplace usage rather than simply generic high e-commerce expenditure.

In [22]:
# Reviewed and adapted Python implementation; verify results and explain each calculation.
# Audit the parser and category coverage without inventing labels for unknown merchants.
audit = df_clean.loc[df_clean.vendor_clean.eq('Uncategorised'), ['Description', 'amount']]
print('Unmapped-vendor audit (expected zero or fewer than five):')
print(audit.head(10).to_string(index=False) if len(audit) else 'No unmapped merchant descriptions.')

weekend_debits = debits[debits.day_of_week.isin(['Saturday', 'Sunday'])]
weekday_debits = debits[~debits.day_of_week.isin(['Saturday', 'Sunday'])]
weekend_days = df_clean.loc[df_clean.date.dt.dayofweek >= 5, 'date'].nunique()
weekday_days = df_clean.loc[df_clean.date.dt.dayofweek < 5, 'date'].nunique()
weekend_daily = weekend_debits.amount.sum() / weekend_days if weekend_days else 0
weekday_daily = weekday_debits.amount.sum() / weekday_days if weekday_days else 0
print(f'\nAverage debit per observed weekend day: Rs. {weekend_daily:,.0f}')
print(f'Average debit per observed weekday: Rs. {weekday_daily:,.0f}')
print('(Denominator: distinct transaction dates appearing in the CSV.)')

marketplace_orders = debits[debits.vendor_clean.isin(['Amazon', 'Flipkart'])]
marketplace_pct = (100 * marketplace_orders.amount.sum() / amount_out) if amount_out else 0
marketplace_regular = len(marketplace_orders) >= 100 and marketplace_pct > 25
print(f'\nCustom archetype THE MARKETPLACE POWER USER: {"MATCH" if marketplace_regular else "NO MATCH"} ({len(marketplace_orders)} purchases and {marketplace_pct:.1f}% of all debits; rules >=100 and >25%)')

next_month_forecast = monthly_spend[['Apr', 'May', 'Jun']].to_numpy(dtype=float).mean(axis=1)
forecast = pd.Series(next_month_forecast, index=monthly_spend.index, name='Jul forecast').sort_values(ascending=False)
print('\nSimple July 2024 forecast: average of Apr, May, Jun (not an ML forecast):')
print(forecast.head(8).round().to_string())

Unmapped-vendor audit (expected zero or fewer than five):
No unmapped merchant descriptions.

Average debit per observed weekend day: Rs. 9,157
Average debit per observed weekday: Rs. 9,252
(Denominator: distinct transaction dates appearing in the CSV.)

Custom archetype THE MARKETPLACE POWER USER: MATCH (123 purchases and 29.5% of all debits; rules >=100 and >25%)

Simple July 2024 forecast: average of Apr, May, Jun (not an ML forecast):
category
E-commerce         100030.0
Investments         42028.0
Food Delivery       20472.0
Restaurants         18593.0
Housing             18000.0
Quick Commerce      15809.0
Fuel                10246.0
Cash Withdrawal     10167.0


## Final deliverable — Screenshot-ready SpendDNA report
This is the final all-in-one text output. It contains more than eight labelled analytical sections, explicit denominators, measured archetypes and computed findings. All numerical values come from running the preceding cells; there are no fabricated target numbers.

In [23]:
# Reviewed and adapted Python implementation; verify results and explain each calculation.
def inr(value):
    return f'Rs. {value:,.0f}'

def ascii_bar(value, maximum, width=22):
    return '#' * max(0, int(round(width * value / maximum))) if maximum > 0 else ''

line = '=' * 82
report = [line, '                 SpendDNA | RAHUL SHARMA | 2024',
          f'      {df_clean.date.min():%d %b %Y} to {df_clean.date.max():%d %b %Y}  |  {len(df_clean):,} transactions  |  synthetic dataset', line]
def section(title):
    report.extend(['', '  ' + title.upper(), '  ' + '-' * 70])

section('01 / Data quality')
report.append(f'  Raw transactions : {original_rows:,}    Duplicates removed : {duplicates_removed}')
report.append(f'  Invalid dates    : {invalid_dates}       Invalid amounts    : {invalid_amounts}')
report.append(f'  Vendors mapped   : {df_clean.vendor_clean.nunique()}       Uncategorised rows : {df_clean.category.eq("Uncategorised").sum()}')

section('02 / Executive summary')
report.append(f'  Credits          : {inr(amount_in):>18}')
report.append(f'  Debits           : {inr(amount_out):>18}')
report.append(f'  Net cash change  : {inr(net_cash_flow):>18}')
report.append(f'  Savings rate     : {savings_rate:>17.1f}%')
report.append(f'  Account entries  : {len(df_clean):>18,}')

section('03 / Leading debit categories (share of ALL debits)')
for cat, spent in by_category.head(8).items():
    portion = share_of_debits.loc[cat]
    report.append(f'  {cat:<18} {ascii_bar(portion, 35):<22} {portion:>5.1f}%   {inr(spent):>15}')
report.append('  Note: P2P / ATM excluded from ranking, but included in % denominator.')

section('04 / Top debit vendors')
for vendor, row in by_vendor.head(7).iterrows():
    report.append(f'  {vendor:<22} {inr(row["sum"]):>17}  ({int(row["count"]):>3} debits)')

section('05 / Monthly debit trend')
peak_month = float(monthly_debit_total.max())
for month, spend in zip(month_labels, monthly_debit_total):
    report.append(f'  {month:<6} {inr(spend):>15}  {ascii_bar(spend, peak_month, width=30)}')
if len(change_percent):
    report.append(f'  Jan-Jun fastest rise: {change_percent.idxmax()} {change_percent.max():+.1f}%')
    report.append(f'  Jan-Jun largest drop: {change_percent.idxmin()} {change_percent.min():+.1f}%')

section('06 / Time-of-day behavior')
report.append(f'  Food delivery 21:00-01:59 : {late_order_pct:>5.1f}% of orders ({len(late_orders)} / {len(food_orders)})')
report.append(f'  Late-night food rupees    : {late_food_spend_pct:>5.1f}% of food-delivery total')
report.append(f'  Cafe 08:00-11:59          : {morning_cafe_pct:>5.1f}% of cafe transactions')
for cat in ['Food Delivery', 'Quick Commerce', 'Cafe']:
    if cat in hour_count.index:
        peaks = hour_count.loc[cat].sort_values(ascending=False).head(3).index.tolist()
        report.append(f'  {cat:<24}: peak hours ' + ', '.join(f'{hour:02d}:00' for hour in peaks))

section('07 / Largest within-category anomalies (z > 2)')
report.append(f'  Flagged debit transactions: {len(anomalies)}  [unusual size, NOT a fraud claim]')
for row in anomalies.head(5).itertuples():
    report.append(f'  {row.date:%d %b %Y}  {row.vendor_clean:<18} {inr(row.amount):>14}  z={row.z_score:5.2f}')

section('08 / Financial personality')
for label, metric in active_archetypes.items():
    report.append(f'  > {label:<29} {metric}')
report.append(f'  > THE MARKETPLACE POWER USER (custom): {"YES" if marketplace_regular else "NO"}  ({len(marketplace_orders)} Amazon/Flipkart debits; {marketplace_pct:.1f}% of total)')

section('09 / Bonus weekly pattern + simple forecast')
report.append(f'  Avg spending per observed weekend day  : {inr(weekend_daily)}')
report.append(f'  Avg spending per observed weekday      : {inr(weekday_daily)}')
report.append(f'  July cash-debit forecast (Apr-Jun mean): {inr(forecast.sum())}')

section('10 / Three measured insights')
report.append(f'  1) Debits exceeded credits by {inr(-net_cash_flow)} ({inr(-net_cash_flow/6)} per month).' if net_cash_flow < 0 else f'  1) Credits exceeded debits by {inr(net_cash_flow)}.')
report.append(f'  2) {late_order_pct:.1f}% of food-delivery orders occurred 21:00-01:59; {late_food_spend_pct:.1f}% of food spend.')
combined_share = category_pct(['Food Delivery', 'Quick Commerce', 'E-commerce'])
report.append(f'  3) Food Delivery + Quick Commerce + E-commerce total {combined_share:.1f}% of all debits.')
report.append(line)
report.append('  Interpretation: Synthetic bank cash-flow data; not a credit score or fraud detector.')
report.append(line)

report_text = '\n'.join(report)
print(report_text)

                 SpendDNA | RAHUL SHARMA | 2024
      01 Jan 2024 to 30 Jun 2024  |  1,310 transactions  |  synthetic dataset

  01 / DATA QUALITY
  ----------------------------------------------------------------------
  Raw transactions : 1,328    Duplicates removed : 18
  Invalid dates    : 0       Invalid amounts    : 0
  Vendors mapped   : 44       Uncategorised rows : 0

  02 / EXECUTIVE SUMMARY
  ----------------------------------------------------------------------
  Credits          :        Rs. 509,774
  Debits           :      Rs. 1,678,901
  Net cash change  :     Rs. -1,169,127
  Savings rate     :            -229.3%
  Account entries  :              1,310

  03 / LEADING DEBIT CATEGORIES (SHARE OF ALL DEBITS)
  ----------------------------------------------------------------------
  E-commerce         ######################  35.4%       Rs. 593,769
  Investments        #########               14.8%       Rs. 248,160
  Restaurants        #####                    7.6%      

## Validation — Assignment rubric checks
These sanity tests check the expected 1,310 rows and 18 duplicates, positive monetary values, coverage of all twelve named core categories, complete vendor classification, and correctly bounded z-score results. The supplied dataset's actual total debits may differ from the illustrative figures in the project handout. **Always show computed results, not the handout's sample numbers.**

In [24]:
# Reviewed and adapted Python implementation; verify results and explain each calculation.
CORE_CATEGORIES = {
    'Food Delivery', 'Quick Commerce', 'E-commerce', 'Transport', 'Cafe', 'Restaurants',
    'Subscriptions', 'Utilities', 'Groceries', 'Investments', 'Fuel', 'Entertainment'
}
assert CORE_CATEGORIES.issubset(set(df_clean['category'])), 'At least one of the 12 core categories is missing.'
assert df_clean['vendor_clean'].eq('Uncategorised').sum() < 5, 'The vendor dictionary needs more keywords.'
assert (df_clean['amount'] >= 0).all(), 'Unexpected negative amount in dataset.'
assert len(ARCHETYPE_RULES) == 9, 'All nine named archetype rules should be evaluated.'
assert (anomalies['z_score'] > 2).all(), 'The anomaly filter is inconsistent.'
assert np.isclose(amount_in - amount_out, net_cash_flow), 'Cash-flow calculation mismatch.'
assert np.isclose(by_category.sum() + debits.loc[debits.category.isin(NON_CONSUMPTION), 'amount'].sum(), amount_out), 'Some debits are not represented in the breakdown.'
print('PASS: parser, vendor mapping, 12 core categories, financial reconciliation, z-scores, nine archetype checks.')
print('Dataset observation: official handout figures are illustrative; numbers above are calculated directly from the accompanying CSV.')

PASS: parser, vendor mapping, 12 core categories, financial reconciliation, z-scores, nine archetype checks.
Dataset observation: official handout figures are illustrative; numbers above are calculated directly from the accompanying CSV.


## Reflection and responsible data use

**What I learned:** Indian bank descriptions contain aliases, prefixes and payment-network noise. Merchant normalization needs ordering and auditing before categorization. The dataset also demonstrates why mixed date formats must be parsed explicitly and why category-wise z-scores are different from a global z-score.

**Limitation:** High z-scores identify unusually large transfers within a category; they do not establish fraud. The account can record investments and cash withdrawals as debits even though those are not equivalent to consumption. Category shares state the denominator to prevent misleading interpretation. The dataset is fictional, and patterns should not be generalized to real individuals.

**AI-assistance disclosure:** Code structure, merchant mapping and narrative formatting were drafted with AI assistance and validated against the provided synthetic CSV. Every executed code cell includes a disclosure comment. I am responsible for understanding and adapting the solution before submission.

**Next improvement:** Add a configurable alias dictionary maintained by the student, defensive tests for edge cases, and support for additional financial-statement CSV layouts without uploading personal financial data.

**Submission:** Upload `SpendDNA_Arathi_A.ipynb` and `rahul_transactions.csv` to Colab, run all cells, share the **viewable notebook link only** in the Unlox Academy form.